# SnapChef - YOLOv8 Edge Vision Training
This notebook trains a YOLOv8 model on a food ingredients dataset and exports it to `.tflite` for use in React Native (`react-native-fast-tflite`).

**v3 changes** (bigger model, more training, crash-resilient):
- Switched from `yolov8n` (nano, 3.3M params) to **`yolov8s`** (small, ~11M params) — meaningfully higher accuracy ceiling. Still fast enough on-device since detection runs once per photo, not continuously on video.
- **100 epochs** instead of 50 — v2's 50-epoch nano run landed at mAP50 ≈ 0.50 and, in real on-device testing, was only confidently right about the single top-scoring class per photo. More capacity + more training targets meaningfully better recall.
- **Backs up the training checkpoint to Drive every 10 epochs**, not just at the end — a run this long has real risk of a Colab disconnect partway through; previously that meant starting over from zero (see v1's `data.yaml` loss). Now a disconnect loses at most ~10 epochs of progress, resumable from Drive.
- (Carried over from v2: Drive backup of `data.yaml`/weights/export, in-Colab sanity-check before download, plain float32 export — no int8 quantization, since that produced an export react-native-fast-tflite's TFLite engine couldn't run.)

## 1. Setup Environment

In [ ]:
!pip install -q ultralytics roboflow


## 2. Mount Google Drive (backup target)
Do this **first**, before anything else — everything downloaded/trained below gets backed up here as we go, so a disconnected runtime never means starting over from zero again.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BACKUP_DIR = '/content/drive/MyDrive/SnapChefVision'
os.makedirs(BACKUP_DIR, exist_ok=True)
print(f"Backups will be saved to: {BACKUP_DIR}")


## 3. Download Dataset from Roboflow
1. Go to Roboflow Universe and find a food ingredients dataset (YOLOv8 format).
2. Click 'Download Dataset' -> 'Show Download Code'.
3. Paste your Roboflow API key and workspace/project details below.

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="mHMwqsFzwVtHmAeu2bTg")
project = rf.workspace("food-recipe-ingredient-images-0gnku").project("food-ingredients-dataset")
version = project.version(4)
dataset = version.download("yolov8")

# Back up data.yaml (the class list) immediately — this is the file that was
# lost entirely last time, forcing the labels to be reverse-engineered from
# Roboflow's API after the fact.
import shutil
shutil.copy(f"{dataset.location}/data.yaml", f"{BACKUP_DIR}/data.yaml")
print(f"Backed up data.yaml to {BACKUP_DIR}/data.yaml")


## 4. Train YOLOv8 Small

**If resuming after a disconnect**: instead of the cells below, run this first to reload the last backed-up checkpoint, then continue training from there —
```python
model = YOLO(f"{BACKUP_DIR}/last_checkpoint.pt")
results = model.train(resume=True)
```
Otherwise (a fresh run), use the cells below as normal.

In [ ]:
from ultralytics import YOLO

# Load the small model — more capacity than the nano variant used before,
# for a meaningfully higher accuracy ceiling. Still fine on-device: this
# runs once per photo (after "Use Photo"), not continuously on live video.
model = YOLO('yolov8s.pt')


### Auto-backup checkpoints to Drive during training
Registers a callback that copies the latest checkpoint (`last.pt`) to Drive every 10 epochs. If the Colab runtime disconnects partway through this long a run, at most ~10 epochs of progress is lost — resume by loading this checkpoint instead of starting over.

In [ ]:
import shutil
from pathlib import Path

CHECKPOINT_BACKUP_EVERY = 10

def backup_checkpoint(trainer):
    if trainer.epoch % CHECKPOINT_BACKUP_EVERY != 0:
        return
    last = Path(trainer.save_dir) / "weights" / "last.pt"
    if last.exists():
        shutil.copy(last, f"{BACKUP_DIR}/last_checkpoint.pt")
        print(f"[checkpoint backup] epoch {trainer.epoch}: synced last.pt to Drive")

model.add_callback('on_train_epoch_end', backup_checkpoint)


In [ ]:
# Train the model. We use imgsz=320 for faster mobile inference.
# Note: dataset.location contains the path to the downloaded data.yaml
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=320,
    batch=16,
    project="snapchef_vision",
    name="edge_model"
)


In [ ]:
# Robust path lookup — different Ultralytics versions nest run folders
# differently (e.g. runs/detect/<project>/<name>/ vs <project>/<name>/
# directly), so find things by glob instead of assuming one fixed layout.
import glob
from pathlib import Path

def find_latest(pattern):
    matches = glob.glob(f"/content/**/{pattern}", recursive=True)
    assert matches, f"Could not find anything matching {pattern}"
    return Path(max(matches, key=lambda p: Path(p).stat().st_mtime))


## 5. Back Up Trained Weights to Drive
Do this immediately after training finishes, before anything else can go wrong — `best.pt` is the actual trained model; everything after this point (export, download) can be re-derived from it without retraining.

In [ ]:
import shutil

weights_src = find_latest("edge_model/weights/best.pt")
weights_backup = f"{BACKUP_DIR}/best.pt"
shutil.copy(weights_src, weights_backup)
print(f"Found trained weights at: {weights_src}")
print(f"Backed up trained weights to {weights_backup}")


## 6. Verify & Test the Model
Let's run the trained model on a test image to make sure it's working perfectly.

In [ ]:
from IPython.display import Image, display

# Grab a random image from the test set
test_images = glob.glob(f"{dataset.location}/test/images/*.jpg")
if len(test_images) > 0:
    test_image = test_images[0]

    # Run inference
    res = model.predict(source=test_image, imgsz=320, save=True, project="snapchef_vision", name="test_inference")

    # Display the result — use the actual save_dir Ultralytics reports back,
    # rather than assuming where it saved.
    display(Image(filename=str(Path(res[0].save_dir) / Path(test_image).name)))
else:
    print("No test images found.")


## 7. Export to TFLite (float32)
Plain float32 — **not** `int8` this time. Deliberately skipping quantization: it's the export path that produced ops `react-native-fast-tflite`'s pinned TensorFlowLiteC (2.17.0) can't execute. Float32 is larger (tens of MB instead of a few MB for a nano model) but that's a non-issue for a phone app, and it's the most universally-supported export TFLite has.

In [ ]:
# Export to TFLite — plain float32, no int8/half quantization
model.export(
    format='tflite',
    imgsz=320,
    int8=False,
    half=False,
)

tflite_path = find_latest("edge_model/weights/*.tflite")
print(f"Exported model: {tflite_path}")


## 8. Sanity-Check the Exported .tflite
Run real inference through the exported TFLite file itself (not the original PyTorch model) on a test image, right here in Colab. If this produces sane-looking detections, the export is good and the file is safe to bring into the app.

In [ ]:
tflite_model = YOLO(tflite_path)
res = tflite_model.predict(source=test_image, imgsz=320, save=True, project="snapchef_vision", name="tflite_test_inference")
display(Image(filename=str(Path(res[0].save_dir) / Path(test_image).name)))

print("\nDetected classes in this test image:")
for r in res:
    for c in r.boxes.cls:
        print(" -", model.names[int(c)])


## 9. Back Up the Exported Model to Drive
Second backup point — in case the download step below fails or gets interrupted, the exported model (and its data.yaml) are already safe.

In [ ]:
import shutil

shutil.copy(tflite_path, f"{BACKUP_DIR}/best_float32.tflite")
print(f"Backed up exported model to {BACKUP_DIR}/best_float32.tflite")


## 10. Download the Model
Downloads both the `.tflite` model and its `data.yaml` (class list) directly to your computer. Put `best_float32.tflite` in `mobile/assets/models/` (replacing `best_int8.tflite`) and `data.yaml` alongside it for reference.

In [ ]:
from google.colab import files

files.download(tflite_path)
files.download(f"{dataset.location}/data.yaml")
